In [0]:
# Importing required libraries
from pyspark.sql.functions import *
from delta.tables import DeltaTable

In [0]:
# Reading master dataset
master_df = spark.read.csv("/Volumes/workspace/celebal/w7_dataset/superstore_master.csv",header=True,inferSchema=True)
master_df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+-------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|  Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+-------+--------+--------+--------+
|   101|CA-2016-152156|11-08-2016|11-11-2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset Col

In [0]:
# Checking schema
master_df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



Cleaning Data


In [0]:
# Handling null values
clean_df = master_df.na.fill("Unknown")

In [0]:
# Removing duplicate records
clean_df = clean_df.dropDuplicates()
print("Total Records:",clean_df.count())
master_df.show(5)

Total Records: 10
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+-------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|  Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+-------+--------+--------+--------+
|   101|CA-2016-152156|11-08-2016|11-11-2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases

Creating Delta Table


In [0]:
# Renaming column names for Delta table
for old_name in clean_df.columns:
    new_name=old_name.replace(" ","_").replace("-","_")
    clean_df=clean_df.withColumnRenamed(old_name,new_name)

print(clean_df.columns)

['Row_ID', 'Order_ID', 'Order_Date', 'Ship_Date', 'Ship_Mode', 'Customer_ID', 'Customer_Name', 'Segment', 'Country', 'City', 'State', 'Postal_Code', 'Region', 'Product_ID', 'Category', 'Sub_Category', 'Product_Name', 'Sales', 'Quantity', 'Discount', 'Profit']


In [0]:
# Checking updated column names
print(clean_df.columns)

['Row_ID', 'Order_ID', 'Order_Date', 'Ship_Date', 'Ship_Mode', 'Customer_ID', 'Customer_Name', 'Segment', 'Country', 'City', 'State', 'Postal_Code', 'Region', 'Product_ID', 'Category', 'Sub_Category', 'Product_Name', 'Sales', 'Quantity', 'Discount', 'Profit']


In [0]:
# Saving data as Delta table
clean_df.write.format("delta").mode("overwrite").save("/Volumes/workspace/celebal/w7_dataset/delta_superstore")

Reading Incremental Dataset


In [0]:
# Reading incremental dataset
increment_df=spark.read.csv("/Volumes/workspace/celebal/w7_dataset/superstore_incremental.csv",header=True,inferSchema=True)
increment_df.show()

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|   101|CA-2016-152156|11-08-2016|11-11-2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

In [0]:
# Renaming column names
for old_name in increment_df.columns:
    new_name=old_name.replace(" ","_").replace("-","_")
    increment_df=increment_df.withColumnRenamed(old_name,new_name)
print(increment_df.columns)

['Row_ID', 'Order_ID', 'Order_Date', 'Ship_Date', 'Ship_Mode', 'Customer_ID', 'Customer_Name', 'Segment', 'Country', 'City', 'State', 'Postal_Code', 'Region', 'Product_ID', 'Category', 'Sub_Category', 'Product_Name', 'Sales', 'Quantity', 'Discount', 'Profit']


MERGE Operation

In [0]:
# Creating Delta table object
delta_table=DeltaTable.forPath(spark,"/Volumes/workspace/celebal/w7_dataset/delta_superstore")

In [0]:
# Applying MERGE operation
(delta_table.alias("target")
.merge(
increment_df.alias("source"),
"target.Row_ID = source.Row_ID"
)
.whenMatchedUpdateAll()
.whenNotMatchedInsertAll()
.execute())
print("Merge operation successfull")

Merge operation successfull


Validating Result

In [0]:
# Reading updated Delta table
final_df=spark.read.format("delta").load("/Volumes/workspace/celebal/w7_dataset/delta_superstore")

In [0]:
# Checking total records
print("Total Records:",final_df.count())

Total Records: 10


In [0]:
# Checking duplicate Row IDs
final_df.groupBy("Row_ID").count().filter(col("count")>1).show()

+------+-----+
|Row_ID|count|
+------+-----+
+------+-----+



Result


In [0]:
# Displaying Final Dataset
final_df.show(10)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row_ID|      Order_ID|Order_Date| Ship_Date|     Ship_Mode|Customer_ID|  Customer_Name|  Segment|      Country|           City|     State|Postal_Code|Region|     Product_ID|       Category|Sub_Category|        Product_Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|   101|CA-2016-152156|11-08-2016|11-11-2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 